# WOSAC Baseline Official-Compliance Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/wosac-sim-agents-experiments/blob/main/experiments/wosac-baseline/notebooks/wosac_baseline_colab.ipynb)

## Objective
- Validate baseline workflow against official Waymo Sim Agents tutorial APIs.
- Generate official `ScenarioRollouts` and validate protobuf structure.
- Compute official metrics and (optionally) package a submission shard tarball.


In [23]:
# Step 1: Repo sync + deterministic runtime bootstrap
import os
import subprocess
import sys

REPO_URL = 'https://github.com/achyutmorang/wosac-sim-agents-experiments.git'
REPO_DIR = '/content/wosac-sim-agents-experiments'

if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '-b', 'main', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
for p in (REPO_DIR, os.path.join(REPO_DIR, 'src')):
    if p not in sys.path:
        sys.path.insert(0, p)

from src.platform import bootstrap_colab_runtime_with_config, wosac_colab_runtime_config
runtime_cfg = wosac_colab_runtime_config(repo_url=REPO_URL, repo_branch='main', repo_dir=REPO_DIR)
bootstrap = bootstrap_colab_runtime_with_config(runtime_cfg)
print('repo_rev:', bootstrap.repo_sync.repo_rev)

if bootstrap.setup.result.get('restart_required', False):
    raise RuntimeError('Runtime restart required. Restart runtime and rerun this cell.')


[repo] existing checkout: /content/wosac-sim-agents-experiments
[drive] /content/drive already mounted
[drive] Verified read/write access: /content/drive/MyDrive/wosac_experiments
[setup] cache hit -> skipping deterministic setup: ok numpy=2.2.6 pandas=2.2.3
[setup] cached ready state at /content/.wosac_setup_cache.json
repo_rev: 2fee292


In [24]:
# Step 2: Ensure official Waymo Sim Agents APIs are available
import importlib
import subprocess
import sys


def _pip_install(args: list[str]) -> None:
    cmd = [sys.executable, '-m', 'pip', *args]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)


def _ensure_waymo_open_dataset() -> None:
    try:
        importlib.import_module('waymo_open_dataset')
        print('waymo_open_dataset already installed.')
        return
    except Exception:
        pass

    py_major, py_minor = sys.version_info[:2]
    if (py_major, py_minor) >= (3, 12):
        # Colab now defaults to Python 3.12 in some runtimes.
        # Official Waymo wheels pin TensorFlow<=2.13, which may not resolve with deps on 3.12.
        print('Python 3.12+ detected. Trying compatibility install (no pinned deps) first ...')
        try:
            _pip_install(['install', '--upgrade', '--no-deps', 'waymo-open-dataset-tf-2-12-0==1.6.7'])
            importlib.import_module('waymo_open_dataset')
            print('Compatibility install succeeded on Python 3.12+.')
            print('If later metric cells fail due dependency conflicts, switch Colab to Python 3.10/3.11 and rerun.')
            return
        except Exception as exc:
            raise RuntimeError(
                f'Runtime Python {py_major}.{py_minor} is incompatible with default Waymo package deps in Colab. '
                'Switch to Python 3.10/3.11 (Runtime -> Change runtime type) and rerun. '
                f'Compatibility-mode install error: {exc}'
            ) from exc

    install_errors: list[str] = []
    candidates = [
        'waymo-open-dataset-tf-2-12-0==1.6.7',
        'waymo-open-dataset-tf-2-12-0==1.6.4',
    ]
    for pkg in candidates:
        try:
            print(f'Installing {pkg} ...')
            _pip_install(['install', '--upgrade', pkg])
            importlib.import_module('waymo_open_dataset')
            print(f'Installed {pkg} successfully.')
            return
        except Exception as exc:
            install_errors.append(f'{pkg}: {exc}')
            print(f'Install failed for {pkg}: {exc}')

    raise RuntimeError('Failed to install waymo_open_dataset.\n' + '\n'.join(install_errors))


_ensure_waymo_open_dataset()

import numpy as np
import tensorflow as tf
from waymo_open_dataset.protos import scenario_pb2
from waymo_open_dataset.protos import sim_agents_submission_pb2
from waymo_open_dataset.utils import trajectory_utils
from waymo_open_dataset.utils.sim_agents import submission_specs
from waymo_open_dataset.wdl_limited.sim_agents_metrics import metrics

print('Python:', sys.version.split()[0])
print('TensorFlow:', tf.__version__)
print('waymo_open_dataset import: OK')


waymo_open_dataset already installed.
Python: 3.12.12
TensorFlow: 2.20.0
waymo_open_dataset import: OK


In [25]:
# Step 3: Load repo config and run context
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

from src.workflows import bootstrap_experiment_pack, load_experiment_config

EXPERIMENT_SLUG = 'wosac-baseline'
bundle = bootstrap_experiment_pack(slug=EXPERIMENT_SLUG, repo_root='.')
cfg = load_experiment_config(slug=EXPERIMENT_SLUG, repo_root='.')
display(bundle.summary_table)

RUN = dict(cfg.get('run', {}))
EVAL = dict(cfg.get('evaluation', {}))
RUN_NAME = str(RUN.get('run_name', 'dev'))
RUN_PREFIX = str(RUN.get('run_prefix', 'wosac_baseline'))
PERSIST_ROOT = str(RUN.get('persist_root', '/content/drive/MyDrive/wosac_experiments'))
RUN_TAG = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
cfg_hash = hashlib.sha256(json.dumps(cfg, sort_keys=True).encode('utf-8')).hexdigest()
persist_run_dir = Path(PERSIST_ROOT) / f'{RUN_PREFIX}_{RUN_NAME}'
persist_run_dir.mkdir(parents=True, exist_ok=True)

print('RUN_TAG:', RUN_TAG)
print('persist_run_dir:', persist_run_dir)


,field,value
0,slug,wosac-baseline
1,title,WOSAC Baseline
2,objective,Establish reproducible Sim Agents baseline met...
3,notebook,experiments/wosac-baseline/notebooks/wosac_bas...
4,workflow,src/workflows/wosac_baseline_flow.py
5,config_file,/content/wosac-sim-agents-experiments/configs/...


RUN_TAG: 20260303T215849Z
persist_run_dir: /content/drive/MyDrive/wosac_experiments/wosac_baseline_dev


In [26]:
# Step 4: Load one scenario shard example from Waymo GCS (official tutorial style)
from pathlib import Path
import os
import subprocess


def _bool_env(name: str, default: bool = False) -> bool:
    text = os.environ.get(name, '').strip().lower()
    if text in {'1', 'true', 'yes', 'y', 'on'}:
        return True
    if text in {'0', 'false', 'no', 'n', 'off'}:
        return False
    return bool(default)


def _active_gcloud_account() -> str:
    try:
        out = subprocess.check_output(
            [
                'bash',
                '-lc',
                "gcloud auth list --filter=status:ACTIVE --format='value(account)'",
            ],
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
        return out.splitlines()[0].strip() if out else ''
    except Exception:
        return ''


def _ensure_colab_gcs_auth() -> None:
    if 'google.colab' not in sys.modules:
        return

    force_auth = _bool_env('WOSAC_FORCE_GCS_AUTH', False)
    if force_auth:
        subprocess.run(['bash', '-lc', 'gcloud auth revoke --all --quiet'], check=False)

    before = _active_gcloud_account()
    if (not before) or force_auth:
        try:
            from google.colab import auth  # type: ignore
            auth.authenticate_user()
            print('Colab auth: OK')
        except Exception as exc:
            print(f'Colab auth skipped or failed: {exc}')

    after = _active_gcloud_account()
    print('Active gcloud account:', after or '<none>')


def _gcs_path_exists(path: str) -> bool:
    if '*' in path:
        return bool(tf.io.gfile.glob(path))
    return bool(tf.io.gfile.exists(path) or tf.io.gfile.glob(path))


def _dedupe_keep_order(values: list[str]) -> list[str]:
    out: list[str] = []
    for v in values:
        t = str(v).strip().rstrip('/')
        if t and t not in out:
            out.append(t)
    return out


def _discover_sample_tfrecord_from_gcs() -> str:
    explicit = os.environ.get('WOSAC_SAMPLE_TFRECORD', '').strip()
    if explicit:
        if explicit.startswith('gs://'):
            if _gcs_path_exists(explicit):
                if '*' in explicit:
                    return sorted(tf.io.gfile.glob(explicit))[0]
                return explicit
            raise FileNotFoundError(f'WOSAC_SAMPLE_TFRECORD GCS path not found: {explicit}')
        p = Path(explicit).expanduser()
        if p.exists() and p.is_file():
            return str(p)
        raise FileNotFoundError(f'WOSAC_SAMPLE_TFRECORD local path not found: {p}')

    root_env = os.environ.get('WOSAC_GCS_DATASET_ROOT', '').strip()
    extra_roots_env = os.environ.get('WOSAC_GCS_CANDIDATE_ROOTS', '').strip()
    split_env = os.environ.get('WOSAC_GCS_SPLIT', 'validation').strip() or 'validation'

    default_roots = [
        'gs://waymo_open_dataset_motion_v_1_3_1/uncompressed/scenario',
        'gs://waymo_open_dataset_motion_v_1_3_1/scenario',
        'gs://waymo_open_dataset_motion_v_1_2_0/uncompressed/scenario',
        'gs://waymo_open_dataset_motion_v_1_2_0/scenario',
    ]
    roots = _dedupe_keep_order(([root_env] if root_env else []) + [r.strip() for r in extra_roots_env.split(',') if r.strip()] + default_roots)

    split_candidates = _dedupe_keep_order([
        split_env,
        'validation',
        'validation_interactive',
        'training',
        'training_20s',
        'testing',
        'testing_interactive',
        'test',
    ])

    pattern_templates = [
        '{root}/{split}/{split}.tfrecord-00000-of-*',
        '{root}/{split}/{split}.tfrecord-*',
        '{root}/{split}/*.tfrecord-*',
        '{root}/{split}/*.tfrecord',
    ]

    tried: list[str] = []
    for root in roots:
        for split in split_candidates:
            for templ in pattern_templates:
                pattern = templ.format(root=root, split=split)
                tried.append(pattern)
                matches = sorted(tf.io.gfile.glob(pattern))
                if matches:
                    print('Resolved GCS shard from pattern:', pattern)
                    return matches[0]

    print('No shard matched expected patterns. Tried top patterns:')
    for row in tried[:8]:
        print(' -', row)

    for root in roots[:3]:
        for split in split_candidates[:3]:
            ls_cmd = f"gcloud storage ls '{root}/{split}/' | head -n 10"
            print('Diagnostic listing:', f'{root}/{split}/')
            subprocess.run(['bash', '-lc', ls_cmd], check=False)

    return ''


_ensure_colab_gcs_auth()
WOSAC_SAMPLE_TFRECORD = _discover_sample_tfrecord_from_gcs()
assert WOSAC_SAMPLE_TFRECORD, (
    'Could not resolve a sample TFRecord from Waymo GCS\n'
    'Set WOSAC_SAMPLE_TFRECORD explicitly to a concrete gs:// shard path, or set WOSAC_GCS_DATASET_ROOT/WOSAC_GCS_SPLIT.\n'
    'Example root: gs://waymo_open_dataset_motion_v_1_3_1/uncompressed/scenario ; split: validation.'
)

print('WOSAC_SAMPLE_TFRECORD:', WOSAC_SAMPLE_TFRECORD)
dataset = tf.data.TFRecordDataset([WOSAC_SAMPLE_TFRECORD]).take(1)
scenario_bytes = next(dataset.as_numpy_iterator())
scenario = scenario_pb2.Scenario.FromString(scenario_bytes)

challenge_type = submission_specs.ChallengeType.SIM_AGENTS
submission_config = submission_specs.get_submission_config(challenge_type)
sim_agent_ids = submission_specs.get_sim_agent_ids(scenario, challenge_type)

print('scenario_id:', scenario.scenario_id)
print('n_simulation_steps:', submission_config.n_simulation_steps)
print('n_rollouts:', submission_config.n_rollouts)
print('sim_agent_count:', len(sim_agent_ids))



Active gcloud account: axemaurer@gmail.com
Resolved GCS shard from pattern: gs://waymo_open_dataset_motion_v_1_3_1/uncompressed/scenario/validation/validation.tfrecord-00000-of-*
WOSAC_SAMPLE_TFRECORD: gs://waymo_open_dataset_motion_v_1_3_1/uncompressed/scenario/validation/validation.tfrecord-00000-of-00150
scenario_id: b85e1bd6cc8e74c0
n_simulation_steps: 80
n_rollouts: 32
sim_agent_count: 10


In [27]:
# Step 5: Simple baseline policy and ScenarioRollouts generation
def _simulate_with_extrapolation(scenario_obj: scenario_pb2.Scenario) -> tuple[trajectory_utils.ObjectTrajectories, tf.Tensor]:
    traj = trajectory_utils.ObjectTrajectories.from_scenario(scenario_obj)
    obj_ids = tf.convert_to_tensor(submission_specs.get_sim_agent_ids(scenario_obj, challenge_type))
    traj = traj.gather_objects_by_id(obj_ids)
    traj = traj.slice_time(start_index=0, end_index=submission_config.current_time_index + 1)

    states = tf.stack([traj.x, traj.y, traj.z, traj.heading], axis=-1)
    n_objects = states.shape[0]
    last_vel = states[:, -1, :3] - states[:, -2, :3]
    last_vel = tf.concat([last_vel, tf.zeros((n_objects, 1), dtype=states.dtype)], axis=-1)

    valid_diff = tf.logical_and(traj.valid[:, -1], traj.valid[:, -2])
    last_vel = tf.where(valid_diff[:, None], last_vel, tf.zeros_like(last_vel))

    noise_scale = 0.01
    max_action = tf.maximum(tf.reduce_max(tf.abs(last_vel), axis=0), 1e-3)
    sim_states = tf.tile(states[None, :, -1:, :], [submission_config.n_rollouts, 1, 1, 1])

    for _ in range(submission_config.n_simulation_steps):
        current_state = sim_states[:, :, -1, :]
        action_noise = tf.random.normal(current_state.shape, mean=0.0, stddev=noise_scale, dtype=current_state.dtype)
        actions = last_vel[None, :, :] + action_noise * max_action
        next_state = current_state + actions
        sim_states = tf.concat([sim_states, next_state[:, :, None, :]], axis=2)

    sim_states = sim_states[:, :, 1:, :]
    return traj, sim_states

def _joint_scene_from_states(states: tf.Tensor, object_ids: tf.Tensor) -> sim_agents_submission_pb2.JointScene:
    arr = states.numpy()
    sims = []
    for i in range(len(object_ids)):
        sims.append(sim_agents_submission_pb2.SimulatedTrajectory(
            center_x=arr[i, :, 0], center_y=arr[i, :, 1], center_z=arr[i, :, 2], heading=arr[i, :, 3], object_id=int(object_ids[i])
        ))
    return sim_agents_submission_pb2.JointScene(simulated_trajectories=sims)

def _scenario_rollouts_from_states(scenario_obj: scenario_pb2.Scenario, states: tf.Tensor, object_ids: tf.Tensor) -> sim_agents_submission_pb2.ScenarioRollouts:
    joint_scenes = [_joint_scene_from_states(states[i], object_ids) for i in range(states.shape[0])]
    return sim_agents_submission_pb2.ScenarioRollouts(joint_scenes=joint_scenes, scenario_id=scenario_obj.scenario_id)

logged_trajectories, simulated_states = _simulate_with_extrapolation(scenario)
scenario_rollouts = _scenario_rollouts_from_states(scenario, simulated_states, logged_trajectories.object_id)
submission_specs.validate_scenario_rollouts(scenario_rollouts, scenario)
print('ScenarioRollouts validation: PASS')


ScenarioRollouts validation: PASS


In [28]:
# Step 6: Official metrics computation (single scenario bundle)
from pathlib import Path
import os


def _safe_parent_from_module_file(value) -> Path | None:
    text = str(value).strip() if value is not None else ''
    if (not text) or text.lower() == 'none':
        return None
    try:
        return Path(text).expanduser().resolve().parent
    except Exception:
        return None


def _load_metrics_config_safe(challenge):
    try:
        return metrics.load_metrics_config(challenge)
    except Exception as exc:
        print('Default metrics config loader failed:', f'{type(exc).__name__}: {exc}')

    from google.protobuf import text_format
    from waymo_open_dataset.protos import sim_agents_metrics_pb2
    import waymo_open_dataset

    candidate_names = [
        'challenge_2025_sim_agents_config.textproto',
        'challenge_2023_sim_agents_config.textproto',
    ]

    candidates = []
    metrics_dir = _safe_parent_from_module_file(getattr(metrics, '__file__', None))
    if metrics_dir is not None:
        candidates.extend([metrics_dir / name for name in candidate_names])

    wod_parent = _safe_parent_from_module_file(getattr(waymo_open_dataset, '__file__', None))
    wod_site = wod_parent.parent if wod_parent is not None else None
    if wod_site is not None:
        candidates.extend([
            wod_site / 'waymo_open_dataset' / 'wdl_limited' / 'sim_agents_metrics' / name
            for name in candidate_names
        ])

    seen = set()
    for c in candidates:
        sc = str(c)
        if sc in seen:
            continue
        seen.add(sc)
        if c.exists() and c.is_file():
            cfg = sim_agents_metrics_pb2.SimAgentMetricsConfig()
            text_format.Parse(c.read_text(encoding='utf-8'), cfg)
            print('Loaded metrics config from:', c)
            return cfg

    if wod_site is not None:
        cwd = Path.cwd()
        try:
            os.chdir(str(wod_site))
            return metrics.load_metrics_config(challenge)
        finally:
            os.chdir(str(cwd))

    raise FileNotFoundError(
        'Unable to locate default SimAgents metrics config textproto; set a valid waymo-open-dataset install.'
    )


metrics_config = _load_metrics_config_safe(challenge_type)
scenario_metrics = metrics.compute_scenario_metrics_for_bundle(metrics_config, scenario, scenario_rollouts)

print('scenario_metrics proto:')
print(scenario_metrics)




Default metrics config loader failed: FileNotFoundError: [Errno 2] No such file or directory: 'waymo_open_dataset/wdl_limited/sim_agents_metrics/challenge_2025_sim_agents_config.textproto'
Loaded metrics config from: /usr/local/lib/python3.12/dist-packages/waymo_open_dataset/wdl_limited/sim_agents_metrics/challenge_2025_sim_agents_config.textproto
scenario_metrics proto:
scenario_id: "b85e1bd6cc8e74c0"
metametric: 0.810497105
average_displacement_error: 6.10950279
linear_speed_likelihood: 0.000481816125
linear_acceleration_likelihood: 0.460194141
angular_speed_likelihood: 0.987114787
angular_acceleration_likelihood: 0.932286143
distance_to_nearest_object_likelihood: 0.0390358269
collision_indication_likelihood: 0.999968827
time_to_collision_likelihood: 0.999648571
distance_to_road_edge_likelihood: 0.752839744
offroad_indication_likelihood: 0.999968827
min_average_displacement_error: 5.98288584
simulated_collision_rate: 0
simulated_offroad_rate: 0
traffic_light_violation_likelihood: 0.9

In [29]:
# Step 7: Optional submission shard packaging (official proto)
CREATE_SUBMISSION_SHARD = False

if CREATE_SUBMISSION_SHARD:
    output_root = persist_run_dir / 'submission'
    output_root.mkdir(parents=True, exist_ok=True)

    shard_submission = sim_agents_submission_pb2.SimAgentsChallengeSubmission(
        scenario_rollouts=[scenario_rollouts],
        submission_type=sim_agents_submission_pb2.SimAgentsChallengeSubmission.SIM_AGENTS_SUBMISSION,
        account_name=os.environ.get('WOSAC_ACCOUNT_NAME', 'your_account@test.com'),
        unique_method_name=os.environ.get('WOSAC_METHOD_NAME', 'wosac_baseline_tutorial_compliant'),
        authors=[os.environ.get('WOSAC_AUTHOR', 'test')],
        affiliation=os.environ.get('WOSAC_AFFILIATION', 'research'),
        description='WOSAC baseline tutorial-compliant shard generation.',
        method_link='https://github.com/achyutmorang/wosac-sim-agents-experiments',
        uses_lidar_data=False,
        uses_camera_data=False,
        uses_public_model_pretraining=False,
        num_model_parameters=os.environ.get('WOSAC_NUM_PARAMS', '0'),
        acknowledge_complies_with_closed_loop_requirement=True,
    )

    shard_binproto = output_root / 'submission.binproto-00000-of-00001'
    with shard_binproto.open('wb') as f:
        f.write(shard_submission.SerializeToString())

    import tarfile
    tar_path = output_root / 'submission.tar.gz'
    with tarfile.open(tar_path, 'w:gz') as tar:
        tar.add(shard_binproto, arcname=shard_binproto.name)

    print('shard_binproto:', shard_binproto)
    print('submission_tar:', tar_path)
else:
    print('CREATE_SUBMISSION_SHARD=False -> skipped')


CREATE_SUBMISSION_SHARD=False -> skipped


In [30]:
# Step 8: Persist compliance artifact summary (Drive)
def _metric_or_none(name: str):
    text = str(scenario_metrics)
    # Best-effort parse for top-level scalar values from text proto string.
    import re
    m = re.search(rf'{name}: ([0-9eE+\-.]+)', text)
    return float(m.group(1)) if m else None

artifact = {
    'run_id': 'wosac_official_compliance_v0',
    'run_tag': RUN_TAG,
    'scenario_id': scenario.scenario_id,
    'n_rollouts': int(submission_config.n_rollouts),
    'n_simulation_steps': int(submission_config.n_simulation_steps),
    'sim_agent_count': len(sim_agent_ids),
    'primary_metric': str(EVAL.get('primary_metric', 'realism_meta_metric')),
    'metrics_best_effort': {
        'realism_meta_metric': _metric_or_none('metametric'),
        'simulated_collision_rate': _metric_or_none('simulated_collision_rate'),
        'simulated_offroad_rate': _metric_or_none('simulated_offroad_rate'),
        'simulated_traffic_light_violation_rate': _metric_or_none('simulated_traffic_light_violation_rate'),
    },
    'official_proto_text': str(scenario_metrics),
    'provenance': {
        'created_utc': datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
        'config_hash': cfg_hash,
        'experiment_slug': EXPERIMENT_SLUG,
        'tutorial_reference': 'waymo-open-dataset/tutorial/tutorial_sim_agents.ipynb',
    },
}

drive_path = persist_run_dir / 'outputs' / 'wosac_official_compliance_v0.json'
drive_path.parent.mkdir(parents=True, exist_ok=True)
drive_path.write_text(json.dumps(artifact, indent=2, sort_keys=True) + '\n', encoding='utf-8')

print('drive_path:', drive_path)


drive_path: /content/drive/MyDrive/wosac_experiments/wosac_baseline_dev/outputs/wosac_official_compliance_v0.json
